# Financial Fraud Detection — Exploratory Data Analysis

**Dataset:** 1,500 labeled financial transactions across 10 countries with 25 features and a binary fraud target.

**Goal:** Understand feature distributions, fraud patterns, and class imbalance before modeling.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside the current session

/kaggle/input/datasets/mubashirsidiki/financial-fraud-labeled-transaction-data-2026/fraud_1500.csv


## 1. Setup & Load

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('/kaggle/input/datasets/mubashirsidiki/financial-fraud-labeled-transaction-data-2026/fraud_1500.csv')
print(f'Shape: {df.shape}')

Shape: (1500, 25)


In [ ]:
df.head()

In [ ]:
df.info()

**Observations:**
- 1,500 rows, 25 columns
- 6 categorical, 19 numeric
- No missing values across any column

In [ ]:
df.isnull().sum()

In [ ]:
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f"Duplicate transaction_ids: {df['transaction_id'].duplicated().sum()}")

Duplicate rows: 0
Duplicate transaction_ids: 0


**Result:** Zero missing values, zero duplicates. Clean dataset.

## 2. Target Variable — Class Imbalance

In [ ]:
fraud_counts = df['fraud_label'].value_counts()
fraud_pct = df['fraud_label'].value_counts(normalize=True).round(4) * 100

print(f'Legitimate: {fraud_counts[0]} ({fraud_pct[0]:.1f}%)')
print(f'Fraudulent: {fraud_counts[1]} ({fraud_pct[1]:.1f}%)')
print(f'Imbalance ratio: 1:{fraud_counts[0]//fraud_counts[1]}')

Legitimate: 1488 (99.2%)
Fraudulent: 12 (0.8%)
Imbalance ratio: 1:124


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='fraud_label', hue='fraud_label', palette=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'])
ax.set_title('Fraud Label Distribution')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height() + 10), ha='center')
plt.tight_layout()
plt.show()

**Key finding:** Only 12 out of 1,500 transactions are fraudulent (0.8%). This is a 1:124 imbalance ratio — techniques like SMOTE, undersampling, or cost-sensitive learning will be essential for modeling.

## 3. Numeric Feature Distributions

In [ ]:
numeric_cols = ['amount', 'avg_amount_30d', 'hour', 'account_age_days',
                'prior_txn_24h', 'prior_txn_7d', 'days_since_last_txn',
                'merchant_risk_score']

df[numeric_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel('Count')

plt.suptitle('Numeric Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- `amount` is heavily right-skewed (skew=4.7, kurtosis=39.6) — median $38.88, 99th percentile $337.01, max $1,006.52
- `avg_amount_30d` follows similar skew
- `hour` is roughly uniform across 0–23
- `account_age_days` is fairly uniform (1–3,650 days)
- `merchant_risk_score` is uniformly distributed (1–100)

## 4. Amount Deep Dive

In [ ]:
print(f'Amount skew: {df["amount"].skew():.2f}')
print(f'Amount kurtosis: {df["amount"].kurtosis():.2f}')
for p in [90, 95, 99]:
    print(f'{p}th percentile: ${df["amount"].quantile(p/100):.2f}')

Amount skew: 4.7
Amount kurtosis: 39.59
90th percentile: $122.48
95th percentile: $172.58
99th percentile: $337.01


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['amount'], bins=50, color='#3498db', edgecolor='white')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['amount']), bins=50, color='#9b59b6', edgecolor='white')
axes[1].set_title('Log-Transformed Amount')
axes[1].set_xlabel('log(1 + Amount)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
df.groupby('fraud_label')['amount'].describe().round(2)

**Observation:** Fraudulent transactions have a similar mean amount ($60.75) vs legitimate ($58.88), but much lower std ($46.45 vs $69.10). Fraud maxes at $152.36 — high-amount outliers are actually legitimate, not fraudulent. Amount alone won't discriminate well.

## 5. Categorical Features

In [ ]:
cat_cols = ['transaction_type', 'merchant_category', 'device_type', 'country', 'credit_score_band']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=axes[i], color='#3498db')
    axes[i].set_title(f'{col} Distribution')
    axes[i].tick_params(axis='x', rotation=45)

axes[5].set_visible(False)
plt.tight_layout()
plt.show()

**Observations:**
- `transaction_type`: Purchases dominate at 61.5% — transfers/cash_outs are riskier but less frequent
- `merchant_category`: Roughly uniform (~140–168 each)
- `device_type`: Mobile 65.7%, desktop 24.4%, tablet 9.9%
- `country`: US leads (24.1%), followed by India (17.7%)
- `credit_score_band`: "good" is most common (35.9%), "poor" least (10.3%)

## 6. Fraud Rate by Category

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    fraud_rate = df.groupby(col)['fraud_label'].mean().sort_values(ascending=False) * 100
    fraud_rate.plot(kind='bar', ax=axes[i], color='#e74c3c', edgecolor='white')
    axes[i].set_title(f'Fraud Rate by {col}')
    axes[i].set_ylabel('Fraud Rate (%)')
    axes[i].tick_params(axis='x', rotation=45)

axes[5].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
for col in cat_cols:
    rate = df.groupby(col)['fraud_label'].mean().mul(100).round(2)
    print(f'\n--- {col} ---')
    print(rate.sort_values(ascending=False).to_string())

**Key patterns:**
- `transaction_type`: bill_payment highest fraud rate (2.0%), purchases lowest (0.65%)
- `country`: AE (2.17%) and IN (1.51%) lead — both are high-traffic or flagged-risk countries
- `credit_score_band`: "excellent" has highest fraud rate (1.86%) — counterintuitive, worth investigating
- AU, DE, FR, UK have zero fraud cases in this sample

## 7. Time-Based Patterns

In [ ]:
df['hour_bin'] = pd.cut(df['hour'], bins=[0,4,8,12,16,20,24],
                        labels=['0-4','4-8','8-12','12-16','16-20','20-24'], right=False)

hour_fraud = df.groupby('hour_bin', observed=True)['fraud_label'].agg(['mean','sum','count'])
hour_fraud.columns = ['fraud_rate','fraud_count','total']
hour_fraud['fraud_rate'] = (hour_fraud['fraud_rate'] * 100).round(2)
print(hour_fraud)

          fraud_rate  fraud_count  total
hour_bin                                  
0-4            2.87            7    244
4-8            0.42            1    238
8-12           0.39            1    257
12-16          0.37            1    268
16-20          0.00            0    244
20-24          0.80            2    249


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
hour_fraud['fraud_rate'].plot(kind='bar', color='#e74c3c', edgecolor='white', ax=ax)
ax.set_title('Fraud Rate by Hour of Day')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xlabel('Hour Bin')
plt.tight_layout()
plt.show()

**Key finding:** The 0–4 AM window has a 2.87% fraud rate — **7x higher** than daytime hours. Late-night transactions are the strongest temporal fraud signal.

## 8. Binary Flags & Risk Indicators

In [ ]:
flags = ['is_weekend', 'is_foreign_txn', 'is_high_risk_country', 'is_new_device']

print('Flag prevalence and fraud rates:\n')
for col in flags:
    prev = df[col].mean() * 100
    rate_0 = df[df[col]==0]['fraud_label'].mean() * 100
    rate_1 = df[df[col]==1]['fraud_label'].mean() * 100
    print(f'{col}:')
    print(f'  Prevalence: {prev:.1f}%')
    print(f'  Fraud rate when 0: {rate_0:.2f}%')
    print(f'  Fraud rate when 1: {rate_1:.2f}%')
    print()

Flag prevalence and fraud rates:

is_weekend:
  Prevalence: 28.6%
  Fraud rate when 0: 0.84%
  Fraud rate when 1: 0.70%

is_foreign_txn:
  Prevalence: 11.5%
  Fraud rate when 0: 0.53%
  Fraud rate when 1: 2.89%

is_high_risk_country:
  Prevalence: 13.3%
  Fraud rate when 0: 0.77%
  Fraud rate when 1: 1.00%

is_new_device:
  Prevalence: 17.0%
  Fraud rate when 0: 0.64%
  Fraud rate when 1: 1.57%


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, col in enumerate(flags):
    rates = df.groupby(col)['fraud_label'].mean() * 100
    rates.plot(kind='bar', ax=axes[i], color=['#2ecc71','#e74c3c'], edgecolor='white')
    axes[i].set_title(f'Fraud Rate by {col}')
    axes[i].set_ylabel('Fraud Rate (%)')
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(['No', 'Yes'], rotation=0)

plt.tight_layout()
plt.show()

**Key patterns:**
- `is_foreign_txn`: 2.89% fraud vs 0.53% for domestic — **5.5x higher**
- `is_new_device`: 1.57% vs 0.64% — **2.5x higher**
- `is_high_risk_country`: modest elevation
- `is_weekend`: no meaningful difference

## 9. Correlation with Fraud

In [ ]:
numeric_only = df.select_dtypes(include='number').columns.drop('fraud_label')
corr = df[numeric_only].corrwith(df['fraud_label']).sort_values(ascending=False)
print(corr.round(4))

is_foreign_txn            0.0847
days_since_last_txn       0.0681
prior_txn_7d              0.0432
is_new_device             0.0390
is_high_risk_country      0.0308
prior_txn_24h             0.0154
chargeback_history        0.0149
customer_tenure_months    0.0068
account_age_days          0.0064
amount                    0.0024
merchant_id               0.0016
avg_amount_30d           -0.0006
failed_login_24h         -0.0056
is_weekend               -0.0072
user_id                  -0.0113
merchant_risk_score      -0.0214
day_of_week              -0.0277
hour                     -0.0660


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr.plot(kind='barh', color=np.where(corr > 0, '#e74c3c', '#3498db'), ax=ax)
ax.set_title('Feature Correlation with Fraud Label')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

**Observations:**
- Top positive correlates: `is_foreign_txn` (+0.085), `days_since_last_txn` (+0.068), `prior_txn_7d` (+0.043)
- `hour` is negatively correlated (-0.066) — fraud clusters in early hours
- `amount` has near-zero correlation (+0.002) — fraud not simply high-value
- All correlations are weak individually — fraud is signaled by feature combinations, not single columns

## 10. Fraud Case Profiles

In [ ]:
fraud = df[df['fraud_label'] == 1]
print(f'Total fraud cases: {len(fraud)}\n')
fraud[['transaction_id','amount','transaction_type','country','hour',
        'is_foreign_txn','is_new_device','chargeback_history',
        'failed_login_24h','merchant_risk_score','prior_txn_24h']]

Total fraud cases: 12



**Profile of a fraudulent transaction:**
- Amounts range $18–$152 — not extreme
- 8 of 12 occurred between midnight and 5 AM
- 5 of 12 involved foreign transactions
- 4 of 12 were from India, 2 from UAE
- 4 of 12 used a new device
- No single feature flags all 12 — it is the combination that matters

## 11. Summary & Takeaways

| Finding | Detail |
|---|---|
| **Class imbalance** | 0.8% fraud (12/1,500), 1:124 ratio |
| **Strongest fraud signals** | Late-night hours (0–4 AM), foreign transactions, new devices |
| **Weakest signals** | Amount alone, weekend flag, merchant risk score |
| **Geographic pattern** | AE and IN have elevated fraud rates |
| **Feature behavior** | Fraud emerges from feature combinations, not individual columns |
| **Data quality** | Zero missing values, zero duplicates |

**Modeling recommendations:**
- Address class imbalance via SMOTE, ADASYN, or cost-sensitive learning
- Consider tree-based models (XGBoost, Random Forest) that handle non-linear interactions
- Feature engineering: create `amount_to_avg_ratio`, combine `is_foreign_txn` + `is_new_device` as interaction terms
- Evaluate with precision-recall AUC, not accuracy (accuracy is 99.2% by predicting all legitimate)